In [1]:
import streamlit as st
import pandas as pd
import requests
import plotly.express as px

st.set_page_config(page_title="🌍 Météo des capitales", layout="wide")

2025-10-17 14:21:15.469 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [2]:
url = "https://restcountries.com/v3.1/all?fields=name,capital,latlng"
r = requests.get(url)
r.raise_for_status()
data = r.json()
data

[{'name': {'common': 'Lithuania',
   'official': 'Republic of Lithuania',
   'nativeName': {'lit': {'official': 'Lietuvos Respublikos',
     'common': 'Lietuva'}}},
  'capital': ['Vilnius'],
  'latlng': [56.0, 24.0]},
 {'name': {'common': 'Chile',
   'official': 'Republic of Chile',
   'nativeName': {'spa': {'official': 'República de Chile',
     'common': 'Chile'}}},
  'capital': ['Santiago'],
  'latlng': [-30.0, -71.0]},
 {'name': {'common': 'Benin',
   'official': 'Republic of Benin',
   'nativeName': {'fra': {'official': 'République du Bénin',
     'common': 'Bénin'}}},
  'capital': ['Porto-Novo'],
  'latlng': [9.5, 2.25]},
 {'name': {'common': 'Falkland Islands',
   'official': 'Falkland Islands',
   'nativeName': {'eng': {'official': 'Falkland Islands',
     'common': 'Falkland Islands'}}},
  'capital': ['Stanley'],
  'latlng': [-51.75, -59.0]},
 {'name': {'common': 'Georgia',
   'official': 'Georgia',
   'nativeName': {'kat': {'official': 'საქართველო', 'common': 'საქართველო'}}},

In [3]:
len(data)
list_countries = [country['name']['common'] for country in data]
list_countries
# now we have our list of countries
list_capitals = [capital for country in data if country["name"]["common"] not in ["Israel", "South Africa"] for capital in country["capital"]]
list_capitals

list_capitals_latlng = [{"capital" : capital, "lat": country["latlng"][0], "long": country["latlng"][1]} for country in data if country["name"]["common"] not in ["Israel", "South Africa"] for capital in country["capital"] ]
list_capitals_latlng

[{'capital': 'Vilnius', 'lat': 56.0, 'long': 24.0},
 {'capital': 'Santiago', 'lat': -30.0, 'long': -71.0},
 {'capital': 'Porto-Novo', 'lat': 9.5, 'long': 2.25},
 {'capital': 'Stanley', 'lat': -51.75, 'long': -59.0},
 {'capital': 'Tbilisi', 'lat': 42.0, 'long': 43.5},
 {'capital': 'Nicosia', 'lat': 35.0, 'long': 33.0},
 {'capital': 'Accra', 'lat': 8.0, 'long': -2.0},
 {'capital': 'Brussels', 'lat': 50.83333333, 'long': 4.0},
 {'capital': 'Havana', 'lat': 21.5, 'long': -80.0},
 {'capital': 'Andorra la Vella', 'lat': 42.5, 'long': 1.5},
 {'capital': 'Buenos Aires', 'lat': -34.0, 'long': -64.0},
 {'capital': 'San Salvador', 'lat': 13.83333333, 'long': -88.91666666},
 {'capital': 'Tashkent', 'lat': 41.0, 'long': 64.0},
 {'capital': 'Belmopan', 'lat': 17.25, 'long': -88.75},
 {'capital': 'Riga', 'lat': 57.0, 'long': 25.0},
 {'capital': 'Yamoussoukro', 'lat': 8.0, 'long': -5.0},
 {'capital': 'Paris', 'lat': 46.0, 'long': 2.0},
 {'capital': 'Saipan', 'lat': 15.2, 'long': 145.75},
 {'capital': 

In [ ]:
liste_pays_meteo = []
for i in range(1,10):# l'idée ici est d'avoir un get des capitales sélectionéée

    capital_dict = list_capitals_latlng[i] 
    capital_name = capital_dict["capital"]
    capital_lat = capital_dict["lat"]
    capital_lng =  capital_dict["long"]

    url2 = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={capital_lat}&longitude={capital_lng}&current=temperature_2m,relative_humidity_2m"
    )

    r2 = requests.get(url2)
    r2.raise_for_status()
    data2 = r2.json()
    data2

    dat = {
    "name" : capital_name,
    "elevation" : data2["elevation"],
    "timezone" : data2["timezone"],
    "temperature_metric" : data2["current_units"]["temperature_2m"],
    "temperature" : data2["current"]["temperature_2m"],
    "humidity_metric" : data2["current_units"]["relative_humidity_2m"],
    "humidity" : data2["current"]["relative_humidity_2m"]
    }


    liste_pays_meteo.append(dat)

liste_pays_meteo





# on va garder timezeone, elevation, temperature_2m, relative_humidity_2m, current["time"],current["temperature_2m"],current["relative_humidity_2m"]

ReadTimeout: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)

In [8]:

df2 = pd.DataFrame(liste_pays_meteo)

df2.head()



,name,elevation,timezone,temperature_metric,temperature,humidity_metric,humidity
0,Santiago,1209.0,GMT,°C,18.2,%,56
1,Porto-Novo,336.0,GMT,°C,31.1,%,54
2,Stanley,0.0,GMT,°C,6.7,%,73
3,Tbilisi,1067.0,GMT,°C,11.8,%,77
4,Nicosia,689.0,GMT,°C,18.5,%,75


In [9]:
fig = px.scatter(
    df2,
    x="temperature",
    y="humidity",
    text="name",  # ajoute le nom de la ville à côté du point
    hover_name="name",  # affiche le nom en survol
    hover_data={
        "temperature": True,
        "humidity": True,
        "elevation": True,
        "timezone": True
    },
    title="🌡️ Température et humidité dans différentes capitales"
)

# Optionnel : ajuster la taille du texte pour qu’il soit lisible
fig.update_traces(textposition="top center", marker=dict(size=10, color="royalblue"))

# Personnalisation des axes
fig.update_layout(
    xaxis_title="Température (°C)",
    yaxis_title="Humidité (%)",
    template="plotly_white"
)

fig.show()


In [ ]:
# On va maintenant encapsuler ça dans différentes apps, pour augmenter le streamlit à chaque étape
# on fera:
#streamlit run app_etape_1.py
# puis
#streamlit run app_etape_2.py
# etc.


# Améliorations : utiliser l'API d'archives d'open-meteo pour afficher l'historique plutot ques les données instantannées.